# 02 — Phân tích lỗi từ kết quả đánh giá

`results/main/report.md` cho biết **trung bình** mỗi hệ thống được bao nhiêu điểm. Notebook này đi vào
**từng mẫu** (`results/main/per_sample.csv`) để hỏi: model sai ở đâu, sai kiểu gì?

1. Kiểm tra số liệu khớp với report.
2. ViT5 + LoRA thắng/thua ViT5 công khai ở dataset nào, trên những mẫu nào?
3. Điểm phụ thuộc thế nào vào độ dài reference và độ dài bản tóm tắt?
4. Model chép câu hay viết lại (novel n-gram)? Viết lại nhiều có làm sai sự thật nhiều hơn không?
5. Lỗi sự thật mà judge bắt được trông như thế nào?
6. Văn bản dài (`results/long_v2`): concat hơn extractive ở những văn bản nào?

**Không cần GPU hay dữ liệu gốc**: chỉ đọc file kết quả đã commit trong repo.
Cần `pip install -r requirements-notebooks.txt` trong `.venv`.

In [ ]:
import ast
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
os.chdir(ROOT)

from vnsum.eval.metrics import syllables  # noqa: E402

RESULTS = Path("results/main")
NAMES = {"lead3": "Lead-3", "vit5_lora": "ViT5 + LoRA", "vit5_public": "ViT5 công khai", "llm_zeroshot": "LLM zero-shot"}
OURS, BASE = "vit5_lora", "vit5_public"

pd.set_option("display.max_colwidth", 300)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})

ps = pd.read_csv(RESULTS / "per_sample.csv", encoding="utf-8-sig", dtype={"id": str})
ps["name"] = ps["system"].map(NAMES).fillna(ps["system"])
ps["ref_len"] = [len(syllables(s)) for s in ps["summary"].fillna("")]
print(ps.shape, "| hệ thống:", ps["system"].unique().tolist())

## 1. Đối chiếu với report

Trung bình tính lại từ từng mẫu phải trùng với `metrics.csv` (dòng `subset == "all"`). Nếu lệch, file đã bị
sửa tay hoặc không cùng một lần chạy.

In [ ]:
metrics = pd.read_csv(RESULTS / "metrics.csv", encoding="utf-8-sig")
recomputed = ps.groupby("system")[["rouge1", "rouge2", "rougeL", "bertscore_f"]].mean()
official = metrics[metrics["subset"] == "all"].set_index("system")[recomputed.columns]
diff = (recomputed - official.loc[recomputed.index]).abs().max().max()
print(f"Lệch lớn nhất so với metrics.csv: {diff:.6f}")
assert diff < 1e-6, "per_sample.csv và metrics.csv không khớp"

summary = ps.groupby("name").agg(
    ROUGE_L=("rougeL", "mean"), BERTScore=("bertscore_f", "mean"), do_dai=("length_syllables", "mean"),
    novel_2gram=("novel_2gram", "mean"), faithfulness=("faithfulness", "mean"), n_judge=("faithfulness", "count"),
).round(2).sort_values("ROUGE_L", ascending=False)
summary

## 2. Thắng/thua ở đâu?

ROUGE-L theo dataset và theo nhóm độ dài văn bản (`is_long`: > 1,024 token).

In [ ]:
by = ps.pivot_table(index="name", columns=["dataset", "is_long"], values="rougeL", aggfunc="mean").round(2)
by.columns = [f"{d} / {'dài' if long else 'ngắn'}" for d, long in by.columns]
by

So cặp **trên cùng mẫu**: hiệu ROUGE-L của ViT5 + LoRA trừ ViT5 công khai. Phân bố cho thấy thắng/thua là do
vài mẫu cực đoan hay do lệch đều trên cả tập.

In [ ]:
pair = ps.pivot_table(index=["id", "dataset"], columns="system", values="rougeL").reset_index()
pair["delta"] = pair[OURS] - pair[BASE]

fig, ax = plt.subplots()
for name, g in pair.groupby("dataset"):
    ax.hist(g["delta"], bins=50, alpha=0.5, label=f"{name}: TB {g['delta'].mean():+.2f}, thắng {100 * (g['delta'] > 0).mean():.0f}% mẫu")
ax.axvline(0, color="k", lw=1)
ax.set(title=f"ROUGE-L: {NAMES[OURS]} − {NAMES[BASE]} (từng mẫu)", xlabel="chênh lệch ROUGE-L")
ax.legend()
plt.show()

Model công khai của VietAI được train trên VietNews, nên trên VietNews nó hơn hẳn. Đây là những mẫu
VietNews mà model của dự án **thua nhiều nhất**: so ba bản tóm tắt cạnh nhau để xem khác ở đâu.

In [ ]:
worst_ids = pair[pair["dataset"] == "vietnews"].nsmallest(5, "delta")["id"]
view = ps[ps["id"].isin(worst_ids) & ps["system"].isin([OURS, BASE])]
side = view.pivot_table(index="id", columns="system", values="prediction", aggfunc="first")
side.insert(0, "reference", ps.drop_duplicates("id").set_index("id").loc[side.index, "summary"])
side.join(pair.set_index("id")["delta"].round(1))

## 3. Độ dài

Bản tóm tắt của ViT5 + LoRA ngắn hơn reference khá nhiều. ROUGE-L theo nhóm độ dài reference cho thấy
model hụt điểm chủ yếu ở các reference dài hay không.

In [ ]:
edges = np.unique(ps["ref_len"].quantile(np.linspace(0, 1, 6)).round().astype(int))
ps["ref_bin"] = pd.cut(ps["ref_len"], edges, include_lowest=True,
                       labels=[f"{a}–{b}" for a, b in zip(edges[:-1], edges[1:])])
fig, axes = plt.subplots(1, 2)
ps.pivot_table(index="ref_bin", columns="name", values="rougeL", aggfunc="mean", observed=True).plot(ax=axes[0], marker="o")
axes[0].set(title="ROUGE-L theo độ dài reference", xlabel="âm tiết của reference (5 nhóm)", ylabel="ROUGE-L")
ours = ps[ps["system"] == OURS]
axes[1].scatter(ours["ref_len"], ours["length_syllables"], s=4, alpha=0.3)
lim = [0, ours[["ref_len", "length_syllables"]].quantile(0.99).max()]
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set(xlim=lim, ylim=lim, title=f"{NAMES[OURS]}: độ dài bản tóm tắt so với reference",
            xlabel="reference (âm tiết)", ylabel="bản tóm tắt (âm tiết)")
plt.tight_layout()
plt.show()
print(f"{NAMES[OURS]} ngắn hơn reference ở {100 * (ours['length_syllables'] < ours['ref_len']).mean():.0f}% mẫu.")

## 4. Chép hay viết lại? Và cái giá của việc viết lại

`novel_2gram`: % bigram của bản tóm tắt không có trong văn bản gốc (0 = chép nguyên văn như Lead-3).
Faithfulness chỉ có trên mẫu judge đã chấm (khoảng 200 mẫu/hệ thống).

In [ ]:
fig, axes = plt.subplots(1, 2)
for name, g in ps.groupby("name"):
    axes[0].hist(g["novel_2gram"], bins=30, range=(0, 100), alpha=0.45, label=name)
axes[0].set(title="% bigram mới", xlabel="novel 2-gram (%)")
axes[0].legend(fontsize=8)

judged = ps.dropna(subset=["faithfulness"]).copy()
judged["novel_bin"] = pd.cut(judged["novel_2gram"], [-0.1, 10, 25, 50, 100], labels=["≤10", "10–25", "25–50", ">50"])
(judged.pivot_table(index="novel_bin", columns="name", values="faithfulness", aggfunc="mean", observed=True)
       .plot(ax=axes[1], marker="o"))
axes[1].set(title="Faithfulness theo mức viết lại", xlabel="novel 2-gram (%)", ylabel="faithfulness (1–5)")
plt.tight_layout()
plt.show()
judged.pivot_table(index="novel_bin", columns="name", values="faithfulness", aggfunc="count", observed=True)

Bảng cuối là **số mẫu** trong mỗi ô: ô ít mẫu thì đường trên hình dao động nhiều, đừng đọc quá mức.

## 5. Lỗi sự thật trông như thế nào?

Các mẫu ViT5 + LoRA bị judge chấm ≤ 2, kèm những ý judge cho là không có trong văn bản (`unsupported_claims`).

In [ ]:
def claims(x) -> str:
    """Cột lưu list dạng Python ("['ý 1', 'ý 2']") -> "ý 1 | ý 2"."""
    if pd.isna(x):
        return ""
    try:
        return " | ".join(map(str, ast.literal_eval(x)))
    except (ValueError, SyntaxError):
        return str(x)

bad = ps[(ps["system"] == OURS) & (ps["faithfulness"] <= 2)].copy()
bad["claims"] = bad["unsupported_claims"].map(claims)
print(f"{len(bad)} / {ps[(ps['system'] == OURS)]['faithfulness'].notna().sum()} mẫu được chấm có faithfulness ≤ 2")
bad[["id", "dataset", "faithfulness", "summary", "prediction", "claims"]].head(8)

## 6. Văn bản dài: concat so với extractive (`results/long_v2`)

Trên 225 văn bản dài, ghép bản tóm tắt từng chunk (`vit5_lora_hier_concat`) hơn lọc câu rồi tóm tắt một lần
(`vit5_lora`). Hơn đều, hay chỉ hơn ở một nhóm?

In [ ]:
long_path = Path("results/long_v2/per_sample.csv")
if long_path.exists():
    lv = pd.read_csv(long_path, encoding="utf-8-sig", dtype={"id": str})
    lp = lv.pivot_table(index=["id", "dataset"], columns="system", values=["rougeL", "length_syllables"]).reset_index()
    d = lp[("rougeL", "vit5_lora_hier_concat")] - lp[("rougeL", "vit5_lora")]
    dl = lp[("length_syllables", "vit5_lora_hier_concat")] - lp[("length_syllables", "vit5_lora")]
    fig, ax = plt.subplots()
    ax.scatter(dl, d, s=10, alpha=0.5)
    ax.axhline(0, color="k", lw=1)
    ax.set(title="Concat − extractive trên từng văn bản dài", xlabel="concat dài hơn (âm tiết)", ylabel="chênh ROUGE-L")
    plt.show()
    print(f"concat hơn ở {100 * (d > 0).mean():.0f}% văn bản; tương quan giữa chênh độ dài và chênh ROUGE-L: "
          f"{np.corrcoef(dl, d)[0, 1]:.2f}")
    display(pd.DataFrame({"dataset": lp["dataset"], "delta": d}).groupby("dataset")["delta"]  # noqa: F821
              .agg(["count", "mean", lambda s: 100 * (s > 0).mean()]).round(2)
              .rename(columns={"<lambda_0>": "% thắng"}))
else:
    print("Chưa có results/long_v2/per_sample.csv")

## Gợi ý đọc kết quả

- Mọi con số ở đây là **mô tả**; khoảng tin cậy và kiểm định nằm trong `report.md` / `significance.csv`.
- Faithfulness chỉ có trên mẫu được judge chấm, và judge cũng có thể sai (xem `results/main/human_eval/`).
- Phát hiện nào đáng giữ lại (một bảng, một phép đo mới) thì chuyển thành hàm trong `src/vnsum/eval/` kèm test,
  để `scripts/evaluate.py` tự tính trong các lần chạy sau.